# 📘 다변량 데이터와 pandas

**다변량 데이터**는 2개 이상의 변수를 동시에 다루는 데이터입니다.
예를 들어 학생의 "키"와 "몸무게"를 함께 분석하는 것이 다변량 분석입니다.

pandas의 **DataFrame**은 표 형태의 데이터를 다루기에 최적입니다.
이 노트북에서는 pandas로 다변량 데이터를 관리하고 그룹별 통계량을 구합니다.

**학습 목표:**
- pandas DataFrame으로 다변량 데이터 관리
- 그룹별 통계량 계산 (groupby)
- 교차집계표 (pivot_table) 작성

## 1. pandas DataFrame — 다변량 데이터 관리

**DataFrame**은 표(표형식) 형태의 데이터 구조입니다.
행(row)과 열(column)이 있어, 엑셀 시트와 비슷하게 생각할 수 있습니다.

| 개념 | 설명 |
|------|------|
| **행(row)** | 하나의 관측치 (예: 한 마리 물고기) |
| **열(column)** | 하나의 변수 (예: 종, 길이) |
| **인덱스** | 행의 이름 (기본: 0, 1, 2, ...) |

In [ ]:
# ┌─────────────────────────────────────────┐
# │  pandas DataFrame 기본                     │
# │  pd.DataFrame() → 데이터프레임 생성          │
# │  딕셔너리 → DataFrame 변환 가능             │
# │  CSV 파일 → pd.read_csv()로 읽기            │
# └─────────────────────────────────────────┘

import pandas as pd
import numpy as np

# CSV 파일에서 데이터 읽기
fish_multi = pd.read_csv("fish_multi.csv")
print("물고기 다변량 데이터:")
print(fish_multi)

# 데이터프레임 정보
print(f"\n형태: {fish_multi.shape}")       # (행, 열)
print(f"열 이름: {list(fish_multi.columns)}")  # ['species', 'length']
print(f"자료형:\n{fish_multi.dtypes}")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  DataFrame 기본 통계량                      │
# │  .describe() → 기초 통계량 요약             │
# │  .mean(), .std(), .min(), .max()            │
# └─────────────────────────────────────────┘

# 기초 통계량 요약
print("기초 통계량 요약:")
print(fish_multi.describe())

# 개별 통계량
print(f"\n길이 평균: {fish_multi['length'].mean():.2f}")
print(f"길이 표준편차: {fish_multi['length'].std(ddof=1):.2f}")
print(f"길이 중앙값: {fish_multi['length'].median():.2f}")

# 데이터 필터링
print(f"\nA종 물고기:")
print(fish_multi[fish_multi['species'] == 'A'])

print(f"\nB종 물고기:")
print(fish_multi[fish_multi['species'] == 'B'])

## 2. 그룹별 통계량 — groupby

**groupby**는 데이터를 그룹으로 나누어 각각 통계량을 계산합니다.
"종별로 물고기 길이의 평균은?" 같은 질문에 답할 수 있습니다.

> 💡 **groupby의 3단계**: **분할**(Split) → **적용**(Apply) → **결합**(Combine)
> 1. 데이터를 그룹별로 **분할**하고
> 2. 각 그룹에 함수를 **적용**하고
> 3. 결과를 **결합**합니다

In [ ]:
# ┌─────────────────────────────────────────┐
# │  groupby — 그룹별 통계량                    │
# │  .groupby('열이름') → 그룹화                 │
# │  .mean()   → 그룹별 평균                    │
# │  .std()    → 그룹별 표준편차                │
# │  .describe() → 그룹별 상세 통계량            │
# └─────────────────────────────────────────┘

# 종별 그룹화
group = fish_multi.groupby("species")

# 그룹별 평균
print("종별 평균:")
print(group.mean())

# 그룹별 표준편차
print("\n종별 표준편차 (불편):")
print(group.std(ddof=1))

# 그룹별 상세 통계량
print("\n종별 상세 통계량:")
print(group.describe())

In [ ]:
# ┌─────────────────────────────────────────┐
# │  groupby 응용 — 여러 통계량 한 번에          │
# │  .agg() → 여러 함수 동시 적용                │
# └─────────────────────────────────────────┘

# 여러 통계량 동시 계산
stats = fish_multi.groupby("species")["length"].agg(["mean", "median", "std", "min", "max"])
print("종별 통계량 요약:")
print(stats)

# 사용자 정의 함수 적용
def range_func(x):
    return x.max() - x.min()

custom = fish_multi.groupby("species")["length"].agg(["mean", "std", range_func])
custom.columns = ["평균", "표준편차", "범위"]
print("\n종별 통계량 (커스텀):")
print(custom)

## 3. 교차집계표 — pivot_table

**교차집계표**(crosstab)는 두 범주형 변수의 관계를 표로 정리합니다.
예: "매장별 색상별 판매량"을 한눈에 볼 수 있습니다.

| | 파란색 | 빨간색 |
|------|--------|--------|
| **도쿄** | 10 | 15 |
| **오사카** | 13 | 9 |

In [ ]:
# ┌─────────────────────────────────────────┐
# │  교차집계표 — pivot_table                   │
# │  pd.pivot_table()로 두 변수의 관계 정리      │
# │  values=값, index=행, columns=열, aggfunc=함수│
# └─────────────────────────────────────────┘

# CSV 파일에서 매장 판매 데이터 읽기
shoes = pd.read_csv("shoes.csv")
print("신발 판매 데이터:")
print(shoes)

# 교차집계표 (피벗 테이블)
cross = pd.pivot_table(
    data=shoes,
    values="sales",
    aggfunc="sum",
    index="store",
    columns="color"
)
print("\n매장별 색상별 판매량:")
print(cross)

# 행별/열별 합계
print(f"\n매장별 총판매: {cross.sum(axis=1).to_dict()}")
print(f"색상별 총판매: {cross.sum(axis=0).to_dict()}")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  pivot_table 응용                           │
# │  aggfunc을 변경하면 다양한 통계량 가능       │
# │  margins=True → 행/열 합계 추가              │
# └─────────────────────────────────────────┘

# 평균 판매량
cross_mean = pd.pivot_table(
    data=shoes,
    values="sales",
    aggfunc="mean",
    index="store",
    columns="color"
)
print("매장별 색상별 평균 판매량:")
print(cross_mean)

# 합계와 함께 (margins)
cross_total = pd.pivot_table(
    data=shoes,
    values="sales",
    aggfunc="sum",
    index="store",
    columns="color",
    margins=True  # 행/열 합계 추가
)
print("\n합계 포함:")
print(cross_total)

## 🎯 연습 문제

1. 다음 데이터로 DataFrame을 만들고 기초 통계량을 구하세요:
   ```python
   data = {"class": ["A","A","A","B","B","B"], "score": [80, 85, 90, 70, 75, 95]}
   ```
2. 위 데이터를 `groupby("class")`로 그룹화하고, 반별 평균과 표준편차를 구하세요.
3. `pd.pivot_table()`을 사용해 반별 과목별 점수표를 만드세요.
4. `agg()`를 사용해 반별로 평균, 중앙값, 최솟값, 최댓값을 한 번에 구하세요.